# 🌤️ Forecast Silver Tables

## 1. Tabla: `weather_daily_silver_forecast`

---

## 📌 Descripción

Tabla a nivel diario que contiene métricas agregadas del clima **forecast (predicción)** por ciudad y fecha.

Cada registro representa:

- 1 ciudad  
- 1 día (fecha del forecast)  
- Última predicción disponible (según `ingestion_time`)  

---

## 📊 Granularidad
   city + date

---

## 🧱 Columnas

| Columna | Tipo | Descripción |
|--------|------|------------|
| city | string | Nombre de la ciudad (proveniente de metadata) |
| date | string / date | Fecha del forecast (día predicho) |
| ingestion_time | timestamp | Momento en que se ingirió la predicción |
| maxtemp_c | double | Temperatura máxima del día (°C) |
| mintemp_c | double | Temperatura mínima del día (°C) |
| avgtemp_c | double | Temperatura promedio del día (°C) |
| avghumidity | long | Humedad promedio (%) |
| totalprecip_mm | double | Precipitación total del día (mm) |
| maxwind_kph | double | Velocidad máxima del viento (km/h) |
| daily_chance_of_rain | long | Probabilidad de lluvia (%) |
| uv | double | Índice UV (radiación solar) |

---

## ⚙️ Lógica aplicada

- Se explota el array:
`data.forecast.forecastday`

- Se genera un nivel intermedio:
1 fila = 1 día por ciudad

- Se seleccionan métricas desde:
`forecastday.day`

- Se utiliza la fecha real del forecast:
day.date

- Se eliminan duplicados utilizando ventana:

```python
partitionBy(city, date)
orderBy(ingestion_time DESC)



In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number,to_timestamp,to_date,hour
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_forecast_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/forecast/")
df_forecast_bronze.show(5)
df_forecast_bronze.printSchema()

In [0]:

df_days = df_forecast_bronze.select(
    col("metadata.ciudad").alias("city"),
    col("date"),
    col("metadata.ingestion_time").alias("ingestion_time"),
    explode(col("data.forecast.forecastday")).alias("day")
)

In [0]:
forecast_daily = df_days.select(
    col("city"),
    col("day.date").alias("date"),
    col("day.day.maxtemp_c").alias("maxtemp_c"),
    col("day.day.mintemp_c").alias("mintemp_c"),
    col("day.day.totalprecip_mm").alias("totalprecip_mm"),
    col("day.day.daily_chance_of_rain").alias("daily_chance_of_rain"),
    col("ingestion_time"),
    col("day.day.uv"),
    col("day.day.avgtemp_c").alias("avgtemp_c"),
    col("day.day.avghumidity").alias("avghumidity"),
    col("day.day.maxwind_kph").alias("maxwind_kph")
)

In [0]:
forecast_daily.printSchema(5)
forecast_daily.show(5)

In [0]:
forecast_daily = forecast_daily.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

forecast_daily_latest = (
    forecast_daily
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
forecast_daily_latest.show()

In [0]:
forecast_daily_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_daily_silver_forecast")

In [0]:
spark.table("weather_daily_silver_forecast").show(5)

# Tabla weather_hour_silver_forecast

# 📌 Descripción

Tabla a nivel horario que contiene métricas del clima forecast (predicción) por ciudad y fecha-hora.

Cada registro representa:
- 1 ciudad
- 1 hora específica del forecast
- Última predicción disponible (según `ingestion_time`)

# 📊 Granularidad

`city` + `time`

**🔑 Clave lógica:** una fila por ciudad y timestamp exacto

# 🧱 Columnas

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `city` | string | Nombre de la ciudad |
| `date` | date | Fecha del forecast (puede derivarse de `time`) |
| `time` | timestamp | Fecha y hora exacta del forecast |
| `hour` | integer | Hora extraída de `time` (0–23) |
| `ingestion_time` | timestamp | Momento en que se ingirió la predicción |
| `temp_c` | double | Temperatura (°C) |
| `precip_mm` | double | Precipitación (mm) |
| `humidity` | long | Humedad (%) |
| `wind_kph` | double | Velocidad del viento (km/h) |
| `cloud` | long | Nubosidad (%) |
| `chance_of_rain` | long | Probabilidad de lluvia (%) |
| `will_it_rain` | long | Indicador binario de lluvia (1 = sí, 0 = no) |

# ⚙️ Lógica aplicada

## 1. Explosión de datos

Se explota el array:
`day.hour`

Generando:
- 1 fila = 1 hora por ciudad por día

## 2. Selección de métricas

Se seleccionan campos desde:
Luego se extrae la hora:
hour("time")

In [0]:
df_hourly = (
    df_days
    .withColumn("hour", explode(col("day.hour")))
)

In [0]:
forecast_hourly = df_hourly.select(
    col("city"),
    col("date"),
    col("hour.time").alias("time"),
    col("hour.feelslike_c").alias("feelslike_c"),
    col("hour.uv").alias("uv"),
    col("hour.temp_c").alias("temp_c"),
    col("hour.precip_mm").alias("precip_mm"),
    col("hour.humidity").alias("humidity"),
    col("hour.wind_kph").alias("wind_kph"),
    col("hour.cloud").alias("cloud"),
    col("hour.chance_of_rain").alias("chance_of_rain"),
    col("hour.will_it_rain").alias("will_it_rain"),
    col("ingestion_time")
)


In [0]:
forecast_hourly.printSchema()
forecast_hourly.show(5)

In [0]:
# 1. Convertir time
forecast_hourly = forecast_hourly.withColumn(
    "time",
    to_timestamp("time", "yyyy-MM-dd HH:mm")
)

# 2. Arreglar date (CLAVE)
forecast_hourly = forecast_hourly.withColumn(
    "date",
    to_date("time")
)

# 3. Extraer hour (opcional)
forecast_hourly = forecast_hourly.withColumn(
    "hour",
    hour("time")
)

# 4. Convertir ingestion_time
forecast_hourly = forecast_hourly.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

In [0]:
forecast_hourly.printSchema()
forecast_hourly.show(5)

In [0]:
window_spec = Window.partitionBy("city", "time").orderBy(desc("ingestion_time"))

forecast_hourly_latest = (
    forecast_hourly
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


In [0]:
forecast_hourly_latest.show(5)

In [0]:
forecast_hourly_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_hourly_silver_forecast")

In [0]:
spark.table("weather_hourly_silver_forecast").show(5)